In [1]:
import os
import sys
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xa
# from pylibs.plot_utils import set_size, setupax_2dmap

In [5]:
# Setup the IODA file and figure output folder
figsavedir = '/glade/derecho/scratch/swei/data_process_tmp'
obspath = "/glade/campaign/ncar/nmmm0072/Data/obs_pandac"
# iodafile = f"{obspath}/viirs_aod_db_n20/obs.viirs_aod_db_n20.2024110100.nc4"
obstype = "viirs_aod_dt_npp"
cdate = "2024110100"
iodafile = f"{obspath}/{obstype}/{cdate}/{obstype}_obs_{cdate}.h5"

varname = "aerosolOpticalDepth"
plotgrp_list = ['ObsValue', 'hofx']
pltwvl = 0.55  # in um
savefig = True
quality = 600
figname_tmpl = '{group}.ociuaa_pace_aod.png'

area_corner = [-90.0, 90.0, -180.0, 180.0]
area = None
proj = ccrs.PlateCarree()

map_grpname = {
    'ObsValue': 'obsv',
    'hofx': 'hofx',
    'ObsBias': 'obsbias',
    'EffectiveError': 'efferr',
    'ObsError': 'obserr',
    'EffectiveQC': 'effqc',
    'PreQC': 'preqc',
}

In [6]:
ds = xa.open_groups(iodafile)

In [7]:
np.unique(ds['/PreQC']['aerosolOpticalDepth'], return_counts=True)

(array([0., 1., 2.]), array([278362,    502, 441250]))

In [5]:
ds = xa.open_groups(iodafile)
lats = ds['/MetaData'].latitude.data
lons = ds['/MetaData'].longitude.data
dt = ds['/MetaData'].dateTime.data
wvl = ds['/MetaData'].sensorCentralWavelength.data

for tmpgrp in plotgrp_list:
    if f"/{tmpgrp}" not in ds.keys() and tmpgrp in plotgrp_list:
        print(f'Remove {tmpgrp} from plotting list')
        plotgrp_list.remove(tmpgrp)

In [13]:
for pltgrp in plotgrp_list:
    print(f'Plotting: {pltgrp}')
    grpname = f'/{pltgrp}'
    grpds = ds[grpname].assign_coords(
        Channel=wvl
    )
    plotdata = grpds[varname].sel(Channel=pltwvl).values
    
    fig, ax, gl = setupax_2dmap(area_corner, area, proj, lbsize=12)
    set_size(8, 5)
    sc = ax.scatter(
        lons,
        lats,
        vmin=0.,
        vmax=2.,
        c=plotdata,
        s=1,
        cmap="jet",
        edgecolors="None",
    )
    cnts = np.count_nonzero(~np.isnan(plotdata))
    ax.set_title(f"Counts= {cnts}")
    plt.colorbar(sc, fraction=0.025, pad=0.04, aspect=20)
    if savefig:
        figname = f"{figsavedir}/{figname_tmpl.format(group=map_grpname[pltgrp])}"
        print(figname)
        fig.savefig(figname, dpi=quality)
        plt.close()

Plotting: ObsValue
/glade/derecho/scratch/swei/data_process_tmp/obsv.ociuaa_pace_aod.png
Plotting: hofx
/glade/derecho/scratch/swei/data_process_tmp/hofx.ociuaa_pace_aod.png
